# Investigating the most common log shapes by hand

In [27]:
import random
import sys
from dataclasses import asdict
from datetime import datetime, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
from logstore.db import connect
from logstore.repositories import LogRepository

pd.set_option("display.max_colwidth", 120)

conn = connect()
logs = LogRepository(conn)

N_LOG_SHAPES_TO_INVESTIGATE = 12

## The most common log shapes

In [28]:
# counts over the whole history of every log shape
df_shapes = pd.read_sql(
    """
    SELECT s.id AS log_shape_id, s.component, s.state, s.template,
           COUNT(*) AS n_logs, COUNT(DISTINCT l.node) AS n_nodes,
           MIN(l.ts) AS first_seen, MAX(l.ts) AS last_seen
    FROM logs l JOIN log_shapes s ON s.id = l.log_shape_id
    GROUP BY s.id
    ORDER BY n_logs DESC
    """,
    conn,
)
total = df_shapes.n_logs.sum()
df_shapes["share"] = (df_shapes.n_logs / total).round(3)
print(f"{len(df_shapes)} log shapes, {total:,} total logs")

top = df_shapes.head(N_LOG_SHAPES_TO_INVESTIGATE)
print(f"the top {N_LOG_SHAPES_TO_INVESTIGATE} hold {top.n_logs.sum() / total:.1%} of the logs")
df_shapes.head(20)

141 log shapes, 411,483 total logs
the top 12 hold 84.3% of the logs


,log_shape_id,component,state,template,n_logs,n_nodes,first_seen,last_seen,share
0,110,switch_module,error,Linkerror event interval expired,80895,16,2004-01-14 16:43:12,2005-05-18 16:40:59,0.197
1,99,node,temperature,ambient=<N>,54119,257,2003-12-26 14:58:30,2006-04-30 07:01:00,0.132
2,96,gige,temperature,warning,49157,8,2003-12-26 12:36:12,2006-04-30 11:23:24,0.119
3,98,gige,temperature,normal,45079,7,2003-12-26 12:41:12,2006-04-30 11:28:24,0.110
4,38,switch_module,fan,Fan speeds ( <N> <N> <N> <N> <N> <N> ),21559,1,2004-09-19 04:42:40,2006-04-10 06:03:16,0.052
5,84,node,status,running,18834,257,2003-12-26 18:42:21,2006-04-27 14:58:04,0.046
6,82,node,status,configured out,16027,256,2003-08-05 11:01:31,2006-04-26 22:18:00,0.039
7,132,switch_module,error,link errors remain current,13538,16,2005-05-19 15:57:24,2006-04-30 09:10:07,0.033
8,7,action,start,boot (command <N>),13075,256,2003-12-26 18:24:10,2006-04-24 21:18:09,0.032
9,6,action,start,wait (command <N>),12689,256,2003-12-26 18:27:10,2006-04-24 21:21:54,0.031


## Demonstrating the available tools before diving in to the incident investigations

### search: logs by node, component, text, log shape and time

In [29]:
hour, day = timedelta(hours=1), timedelta(days=1)

# the anchor for the examples below: the newest "domain panic" log of March 2006
anchor = logs.search(datetime(2006, 3, 1), datetime(2006, 3, 31), text="domain panic", limit=1)[0]
display([anchor])

search_results = logs.search(anchor.ts - day, anchor.ts + hour, node=anchor.node, limit=30)
display(pd.DataFrame([asdict(r) for r in search_results]))

[Log(id=41359, event_id=428082, node='node-D7', component='clusterfilesystem', state='fdmn.panic', time=1143670270.0, flag=1, message='ServerFileSystem: An ServerFileSystem domain panic has occurred on storage911', bad_time=False, ts=datetime.datetime(2006, 3, 29, 22, 11, 10), original_log='428082 node-D7 clusterfilesystem fdmn.panic 1143670270 1 ServerFileSystem: An ServerFileSystem domain panic has occurred on storage911', log_shape_id=36, n_copies=1)]

,id,event_id,node,component,state,time,flag,message,bad_time,ts,original_log,log_shape_id,n_copies
0,41359,428082,node-D7,clusterfilesystem,fdmn.panic,1.143670e+09,1,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage911,False,2006-03-29 22:11:10,428082 node-D7 clusterfilesystem fdmn.panic 1143670270 1 ServerFileSystem: An ServerFileSystem domain panic has occu...,36,1
1,77762,428035,node-D7,clusterfilesystem,clusterfilesystem.not_served,1.143656e+09,1,ClusterFileSystem: ServerFileSystem domain storage911 is no longer served by node node-224,False,2006-03-29 18:14:22,428035 node-D7 clusterfilesystem clusterfilesystem.not_served 1143656062 1 ClusterFileSystem: ServerFileSystem domai...,74,1
2,41360,428033,node-D7,clusterfilesystem,fdmn.panic,1.143656e+09,1,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage911,False,2006-03-29 18:06:32,428033 node-D7 clusterfilesystem fdmn.panic 1143655592 1 ServerFileSystem: An ServerFileSystem domain panic has occu...,36,1
3,77763,428026,node-D7,clusterfilesystem,clusterfilesystem.not_served,1.143654e+09,1,ClusterFileSystem: ServerFileSystem domain storage911 is no longer served by node node-224,False,2006-03-29 17:46:53,428026 node-D7 clusterfilesystem clusterfilesystem.not_served 1143654413 1 ClusterFileSystem: ServerFileSystem domai...,74,1
4,69871,428015,node-D7,clusterfilesystem,fdmn.full,1.143654e+09,1,ServerFileSystem: ServerFileSystem domain root8_local is full,False,2006-03-29 17:43:08,428015 node-D7 clusterfilesystem fdmn.full 1143654188 1 ServerFileSystem: ServerFileSystem domain root8_local is full,59,1
5,41361,428014,node-D7,clusterfilesystem,fdmn.panic,1.143654e+09,1,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage911,False,2006-03-29 17:43:03,428014 node-D7 clusterfilesystem fdmn.panic 1143654183 1 ServerFileSystem: An ServerFileSystem domain panic has occu...,36,1
6,69872,428007,node-D7,clusterfilesystem,fdmn.full,1.143654e+09,1,ServerFileSystem: ServerFileSystem domain root9_local is full,False,2006-03-29 17:35:30,428007 node-D7 clusterfilesystem fdmn.full 1143653730 1 ServerFileSystem: ServerFileSystem domain root9_local is full,59,1
7,77766,427733,node-D7,clusterfilesystem,clusterfilesystem.not_served,1.143653e+09,1,ClusterFileSystem: ServerFileSystem domain storage911 is no longer served by node node-224,False,2006-03-29 17:28:38,427733 node-D7 clusterfilesystem clusterfilesystem.not_served 1143653318 1 ClusterFileSystem: ServerFileSystem domai...,74,1
8,41362,427731,node-D7,clusterfilesystem,fdmn.panic,1.143653e+09,1,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage911,False,2006-03-29 17:25:18,427731 node-D7 clusterfilesystem fdmn.panic 1143653118 1 ServerFileSystem: An ServerFileSystem domain panic has occu...,36,1
9,77767,427726,node-D7,clusterfilesystem,clusterfilesystem.not_served,1.143653e+09,1,ClusterFileSystem: ServerFileSystem domain storage911 is no longer served by node node-224,False,2006-03-29 17:24:55,427726 node-D7 clusterfilesystem clusterfilesystem.not_served 1143653095 1 ClusterFileSystem: ServerFileSystem domai...,74,1


### count_by: log counts per node, component, log shape or hour

In [47]:
window = (anchor.ts - day, anchor.ts + day)

display(pd.DataFrame(logs.count_by("node", *window, text="domain panic"), columns=["node", "n_logs"]))

by_shape = pd.DataFrame(logs.count_by("log_shape", *window, node=anchor.node), columns=["log_shape_id", "n_logs"])
display(by_shape.merge(df_shapes[["log_shape_id", "component", "state", "template"]], on="log_shape_id"))

display(pd.DataFrame(logs.count_by("component", *window), columns=["component", "n_logs"]))

display(pd.DataFrame(logs.count_by("hour", *window, node=anchor.node), columns=["hour", "n_logs"]).sort_values("hour"))

,node,n_logs
0,node-D7,8


,log_shape_id,n_logs,component,state,template
0,74,72,clusterfilesystem,clusterfilesystem.not_served,ClusterFileSystem: ServerFileSystem domain storage<N> is no longer served by node <NODE>
1,36,8,clusterfilesystem,fdmn.panic,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage<N>
2,29,4,clusterfilesystem,clusterfilesystem.no_server,ClusterFileSystem: There is no server for ServerFileSystem domain storage<N>
3,59,2,clusterfilesystem,fdmn.full,ServerFileSystem: ServerFileSystem domain root<N>_local is full
4,60,1,clusterfilesystem,fdmn.full,ServerFileSystem: ServerFileSystem domain cluster_root is full


,component,n_logs
0,node,606
1,gige,328
2,clusterfilesystem,95
3,switch_module,73
4,partition,20
5,action,10
6,unix.hw,8
7,boot_cmd,6
8,domain,2


,hour,n_logs
2,2006-03-29 15,2
5,2006-03-29 16,1
1,2006-03-29 17,15
3,2006-03-29 18,2
6,2006-03-29 22,1
4,2006-03-30 14,2
0,2006-03-30 17,64


### node_timeline: one node's history, with runs of the same log shape collapsed

In [31]:
timeline = logs.node_timeline(anchor.node, anchor.ts - day, anchor.ts)
print(f"{len(timeline)} entries for {anchor.node} in the 24 hours before the anchor")
pd.DataFrame([asdict(e) for e in timeline])

20 entries for node-D7 in the 24 hours before the anchor


,component,state,template,log_shape_id,first_ts,last_ts,count
0,clusterfilesystem,clusterfilesystem.no_server,ClusterFileSystem: There is no server for ServerFileSystem domain storage<N>,29,2006-03-29 15:07:54,2006-03-29 15:08:05,2
1,clusterfilesystem,fdmn.panic,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage<N>,36,2006-03-29 16:07:35,2006-03-29 16:07:35,1
2,clusterfilesystem,clusterfilesystem.not_served,ClusterFileSystem: ServerFileSystem domain storage<N> is no longer served by node <NODE>,74,2006-03-29 17:03:46,2006-03-29 17:03:46,1
3,clusterfilesystem,fdmn.panic,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage<N>,36,2006-03-29 17:11:40,2006-03-29 17:11:40,1
4,clusterfilesystem,clusterfilesystem.not_served,ClusterFileSystem: ServerFileSystem domain storage<N> is no longer served by node <NODE>,74,2006-03-29 17:12:57,2006-03-29 17:12:57,1
5,clusterfilesystem,fdmn.full,ServerFileSystem: ServerFileSystem domain cluster_root is full,60,2006-03-29 17:16:11,2006-03-29 17:16:11,1
6,clusterfilesystem,clusterfilesystem.not_served,ClusterFileSystem: ServerFileSystem domain storage<N> is no longer served by node <NODE>,74,2006-03-29 17:21:55,2006-03-29 17:21:55,1
7,clusterfilesystem,fdmn.panic,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage<N>,36,2006-03-29 17:22:24,2006-03-29 17:22:24,1
8,clusterfilesystem,clusterfilesystem.not_served,ClusterFileSystem: ServerFileSystem domain storage<N> is no longer served by node <NODE>,74,2006-03-29 17:23:22,2006-03-29 17:23:22,1
9,clusterfilesystem,fdmn.panic,ServerFileSystem: An ServerFileSystem domain panic has occurred on storage<N>,36,2006-03-29 17:23:43,2006-03-29 17:23:43,1


### stats: log count, node count, first and last seen, with the same filters

In [32]:
print("the anchor's node in the day before it:", logs.stats(anchor.ts - day, anchor.ts, node=anchor.node))

WINDOWS = {"1h": timedelta(hours=1), "1d": timedelta(days=1), "7d": timedelta(days=7), "30d": timedelta(days=30)}


def stats_table(log):
    rows = []
    for name, window in WINDOWS.items():
        filter_params = dict(start=log.ts - window, end=log.ts, log_shape_id=log.log_shape_id)
        cluster, node = logs.stats(**filter_params), logs.stats(node=log.node, **filter_params)
        rows.append({"window": name, "n_logs.cluster": cluster.n_logs, "n_nodes.cluster": cluster.n_nodes,
                     "n_logs.node": node.n_logs})
    return pd.DataFrame(rows)


stats_table(anchor)

the anchor's node in the day before it: LogStats(n_logs=21, n_nodes=1, first_seen='2006-03-29 15:07:54', last_seen='2006-03-29 22:11:10')


,window,n_logs.cluster,n_nodes.cluster,n_logs.node
0,1h,1,1,1
1,1d,8,1,8
2,7d,9,1,9
3,30d,9,1,9


## One random log for each of the 12 most common log shapes

In [48]:
def random_log(log_shape_id, rng):
    # sampling is for this notebook only: choose a row straight from the table, then read it through search
    rows = conn.execute(
        "SELECT id, ts FROM logs WHERE log_shape_id = ? AND ts IS NOT NULL ORDER BY ts, id", (log_shape_id,)
    ).fetchall()
    log_id, ts = rng.choice(rows)
    ts = datetime.fromisoformat(ts)
    return next(log for log in logs.search(ts, ts, log_shape_id=log_shape_id) if log.id == log_id)


rng = random.Random(0)
cases = [(shape, random_log(shape.log_shape_id, rng)) for shape in top.itertuples()]


def show(number):
    shape, log = cases[number - 1]
    print(f"log shape {shape.log_shape_id}: {shape.component} | {shape.state} | {shape.template}")
    print(f"{shape.n_logs:,} logs ({shape.share:.1%}) from {shape.n_nodes} nodes, {shape.first_seen} to {shape.last_seen}")
    print(f"log {log.id}: {log.ts} | {log.node} | {log.message}")
    return shape, log


pd.DataFrame(
    [{"number": i, "template": shape.template, **asdict(log)} for i, (shape, log) in enumerate(cases, start=1)]
)

,number,template,id,event_id,node,component,state,time,flag,message,bad_time,ts,original_log,log_shape_id,n_copies
0,1,Linkerror event interval expired,341909,272454,Interconnect-0N02,switch_module,error,1.079615e+09,1,Linkerror event interval expired,False,2004-03-18 13:08:55,272454 Interconnect-0N02 switch_module error 1079615335 1 Linkerror event interval expired,110,1
1,2,ambient=<N>,274807,397274,node-77,node,temperature,1.142528e+09,1,ambient=26,False,2006-03-16 16:53:28,397274 node-77 node temperature 1142528008 1 ambient=26,99,1
2,3,warning,208661,124215,gige3,gige,temperature,1.111286e+09,1,warning,False,2005-03-20 02:26:10,124215 gige3 gige temperature 1111285570 1 warning,96,1
3,4,normal,138735,34965,gige7,gige,temperature,1.076506e+09,1,normal,False,2004-02-11 13:19:12,34965 gige7 gige temperature 1076505552 1 normal,98,1
4,5,Fan speeds ( <N> <N> <N> <N> <N> <N> ),50422,41953,Interconnect-0N00,switch_module,fan,1.125932e+09,1,Fan speeds ( 3552 3534 3375 **** 3515 3479 ),False,2005-09-05 14:56:30,41953 Interconnect-0N00 switch_module fan 1125932190 1 Fan speeds ( 3552 3534 3375 **** 3515 3479 ),38,1
5,6,running,125546,206751,node-194,node,status,1.131561e+09,1,running,False,2005-11-09 18:30:40,206751 node-194 node status 1131561040 1 running,84,1
6,7,configured out,105689,387046,node-240,node,status,1.087501e+09,1,configured out,False,2004-06-17 19:32:01,387046 node-240 node status 1087500721 1 configured out,82,1
7,8,link errors remain current,379250,228944,Interconnect-1N00,switch_module,error,1.132350e+09,1,link errors remain current,False,2005-11-18 21:38:39,228944 Interconnect-1N00 switch_module error 1132349919 1 link errors remain current,132,1
8,9,boot (command <N>),29619,459874,node-98,action,start,1.145552e+09,1,boot (command 4220),False,2006-04-20 16:47:15,459874 node-98 action start 1145551635 1 boot (command 4220),7,1
9,10,wait (command <N>),12366,257391,node-94,action,start,1.079070e+09,1,wait (command 2616),False,2004-03-12 05:46:52,257391 node-94 action start 1079070412 1 wait (command 2616),6,1


## Investigations

### 1. `Linkerror event interval expired` (switch_module, error)

In [34]:
shape, log = show(1)

log shape 110: switch_module | error | Linkerror event interval expired
80,895 logs (19.7%) from 16 nodes, 2004-01-14 16:43:12 to 2005-05-18 16:40:59
log 341909: 2004-03-18 13:08:55 | Interconnect-0N02 | Linkerror event interval expired


### 2. `ambient=<N>` (node, temperature)

In [35]:
shape, log = show(2)

log shape 99: node | temperature | ambient=<N>
54,119 logs (13.2%) from 257 nodes, 2003-12-26 14:58:30 to 2006-04-30 07:01:00
log 274807: 2006-03-16 16:53:28 | node-77 | ambient=26


### 3. `warning` (gige, temperature)

In [36]:
shape, log = show(3)

log shape 96: gige | temperature | warning
49,157 logs (11.9%) from 8 nodes, 2003-12-26 12:36:12 to 2006-04-30 11:23:24
log 208661: 2005-03-20 02:26:10 | gige3 | warning


### 4. `normal` (gige, temperature)

In [37]:
shape, log = show(4)

log shape 98: gige | temperature | normal
45,079 logs (11.0%) from 7 nodes, 2003-12-26 12:41:12 to 2006-04-30 11:28:24
log 138735: 2004-02-11 13:19:12 | gige7 | normal


### 5. `Fan speeds ( <N> <N> <N> <N> <N> <N> )` (switch_module, fan)

In [38]:
shape, log = show(5)

log shape 38: switch_module | fan | Fan speeds ( <N> <N> <N> <N> <N> <N> )
21,559 logs (5.2%) from 1 nodes, 2004-09-19 04:42:40 to 2006-04-10 06:03:16
log 50422: 2005-09-05 14:56:30 | Interconnect-0N00 | Fan speeds ( 3552 3534 3375 **** 3515 3479 )


### 6. `running` (node, status)

In [39]:
shape, log = show(6)

log shape 84: node | status | running
18,834 logs (4.6%) from 257 nodes, 2003-12-26 18:42:21 to 2006-04-27 14:58:04
log 125546: 2005-11-09 18:30:40 | node-194 | running


### 7. `configured out` (node, status)

In [40]:
shape, log = show(7)

log shape 82: node | status | configured out
16,027 logs (3.9%) from 256 nodes, 2003-08-05 11:01:31 to 2006-04-26 22:18:00
log 105689: 2004-06-17 19:32:01 | node-240 | configured out


### 8. `link errors remain current` (switch_module, error)

In [41]:
shape, log = show(8)

log shape 132: switch_module | error | link errors remain current
13,538 logs (3.3%) from 16 nodes, 2005-05-19 15:57:24 to 2006-04-30 09:10:07
log 379250: 2005-11-18 21:38:39 | Interconnect-1N00 | link errors remain current


### 9. `boot (command <N>)` (action, start)

In [42]:
shape, log = show(9)

log shape 7: action | start | boot (command <N>)
13,075 logs (3.2%) from 256 nodes, 2003-12-26 18:24:10 to 2006-04-24 21:18:09
log 29619: 2006-04-20 16:47:15 | node-98 | boot  (command 4220)


### 10. `wait (command <N>)` (action, start)

In [43]:
shape, log = show(10)

log shape 6: action | start | wait (command <N>)
12,689 logs (3.1%) from 256 nodes, 2003-12-26 18:27:10 to 2006-04-24 21:21:54
log 12366: 2004-03-12 05:46:52 | node-94 | wait  (command 2616)


### 11. `NIFF: node <NODE> has detected an available network connection on network <N>.<N>.<N>.<N> via interface alt<N>` (unix.hw, net.niff.up)

In [44]:
shape, log = show(11)

log shape 138: unix.hw | net.niff.up | NIFF: node <NODE> has detected an available network connection on network <N>.<N>.<N>.<N> via interface alt<N>
11,455 logs (2.8%) from 256 nodes, 2003-12-26 18:27:18 to 2006-04-27 14:44:57
log 401240: 2005-03-26 19:32:08 | node-168 | NIFF: node node-168 has detected an available network connection on network 5.5.224.0 via interface alt0


### 12. `not responding` (node, status)

In [45]:
shape, log = show(12)

log shape 85: node | status | not responding
10,628 logs (2.6%) from 257 nodes, 2003-12-26 21:07:30 to 2006-04-26 22:17:28
log 106433: 2004-07-15 12:17:53 | node-221 | not responding
